# 01 · Frequentist z-test — `foe.frequentist`

**FOE surfaces:** `pages/experiment_analysis.py`, `pages/small_sample_analysis.py` (z-test fallback),
`utility/automation_engine.py`.

**What is validated**

| FOE function | R reference |
|---|---|
| `FrequentistEngine.run_synthesis` — unpooled z-test p-value, SE, CI, incl. φ (`reduction_factor`) and one-sided tests | `glm(cbind(x, n-x) ~ group, binomial(link = "identity"))`: the Wald SE of the group coefficient is exactly the unpooled SE; `summary(fit, dispersion = φ)` scales it by √φ |
| `compute_interval_difference` (two-sided) | `DescTools::BinomDiffCI(method = "wald")` |
| `apply_sidak` + A/B/n decisions | Šidák: `1 - (1 - α)^(1/m)` applied to the R p-values |
| `FrequentistEngine.estimate_monetary_impact_per_variant` | `msm::deltamethod` on `p_chal·aov_chal − p_ctrl·aov_ctrl` |
| `FrequentistEngine.run_lin_adjustment` — ATE, HC3 SE, p-value, CI, relative lift, Šidák decisions | `lm(y ~ variant * cc)` + `sandwich::vcovHC(type = "HC3")` |

In [1]:
import os
os.environ.setdefault("RPY2_CFFI_MODE", "ABI")  # use rpy2 ABI mode; avoids a noisy API-mode probe
%load_ext rpy2.ipython

import numpy as np
import pandas as pd

from foe.core.models import AlternativeHypothesis as Alt, ExperimentInput
from foe.frequentist.confidence import compute_interval_difference
from foe.frequentist.operations import FrequentistEngine, apply_sidak
from validation_helpers import Checks

checks = Checks()

## 1. Two-arm z-test across scenarios, tails and φ

In [2]:
scenarios = [
    ("typical",   10_000, 320, 10_050, 365),
    ("low rate",  50_000,  60, 49_800,  85),
    ("high rate",  2_000, 900,  2_100, 990),
    ("negative",   8_000, 400,  8_000, 340),
]
conf = 0.95

rows = []
for name, n_c, x_c, n_v, x_v in scenarios:
    for alt in ["two-sided", "greater", "less"]:
        for phi in [1.0, 0.6, 1.7]:
            res = FrequentistEngine().run_synthesis(ExperimentInput(
                visitors=[n_c, n_v], conversions=[x_c, x_v],
                alternative=Alt(alt), confidence_level=conf, reduction_factor=phi,
            ))[0]
            rows.append(dict(scenario=name, alternative=alt, phi=phi,
                             n_c=n_c, x_c=x_c, n_v=n_v, x_v=x_v,
                             se=res.standard_error, p_value=res.p_value,
                             ci_low=res.ci_diff[0], ci_high=res.ci_diff[1]))
py = pd.DataFrame(rows)
py.head()

,scenario,alternative,phi,n_c,x_c,n_v,x_v,se,p_value,ci_low,ci_high
0,typical,two-sided,1.0,10000,320,10050,365,0.002565,0.092283,-0.000709,0.009346
1,typical,two-sided,0.6,10000,320,10050,365,0.001987,0.029754,0.000424,0.008213
2,typical,two-sided,1.7,10000,320,10050,365,0.003345,0.196646,-0.002237,0.010874
3,typical,greater,1.0,10000,320,10050,365,0.002565,0.046142,0.000099,inf
4,typical,greater,0.6,10000,320,10050,365,0.001987,0.014877,0.001050,inf


In [3]:
%%R -i py -i conf -o r_out
alpha <- 1 - conf
r_out <- do.call(rbind, lapply(seq_len(nrow(py)), function(i) {
  s <- py[i, ]
  d <- data.frame(group = factor(c("control", "variant")),
                  x = c(s$x_c, s$x_v), n = c(s$n_c, s$n_v))
  fit <- glm(cbind(x, n - x) ~ group, family = binomial(link = "identity"), data = d)
  co  <- coef(summary(fit, dispersion = s$phi))["groupvariant", ]
  est <- co[["Estimate"]]; se <- co[["Std. Error"]]; z <- est / se
  p <- switch(s$alternative,
              "two-sided" = 2 * pnorm(-abs(z)),
              "greater"   = pnorm(z, lower.tail = FALSE),
              "less"      = pnorm(z))
  ci <- switch(s$alternative,
               "two-sided" = est + c(-1, 1) * qnorm(1 - alpha / 2) * se,
               "greater"   = c(est - qnorm(1 - alpha) * se, Inf),
               "less"      = c(-Inf, est + qnorm(1 - alpha) * se))
  data.frame(se = se, p_value = p, ci_low = ci[1], ci_high = ci[2])
}))

In [4]:
for i, row in py.iterrows():
    tag = f"{row.scenario} | {row.alternative} | phi={row.phi}"
    r = r_out.iloc[i]
    checks.add(f"SE      {tag}", row.se, r.se, rtol=1e-6)
    checks.add(f"p-value {tag}", row.p_value, r.p_value, atol=1e-9, rtol=1e-6)
    checks.add(f"CI low  {tag}", row.ci_low, r.ci_low, rtol=1e-6)
    checks.add(f"CI high {tag}", row.ci_high, r.ci_high, rtol=1e-6)

## 2. Two-sided CI vs `DescTools::BinomDiffCI(method = "wald")`

In [5]:
two_sided = py[(py.alternative == "two-sided") & (py.phi == 1.0)].reset_index(drop=True)

In [6]:
%%R -i two_sided -i conf -o r_wald
suppressPackageStartupMessages(library(DescTools))
r_wald <- do.call(rbind, lapply(seq_len(nrow(two_sided)), function(i) {
  s <- two_sided[i, ]
  ci <- BinomDiffCI(s$x_v, s$n_v, s$x_c, s$n_c, conf.level = conf, method = "wald")
  data.frame(ci_low = ci[1, "lwr.ci"], ci_high = ci[1, "upr.ci"])
}))

In [7]:
for i, row in two_sided.iterrows():
    checks.add(f"Wald CI low  {row.scenario}", row.ci_low, r_wald.ci_low.iloc[i], rtol=1e-8)
    checks.add(f"Wald CI high {row.scenario}", row.ci_high, r_wald.ci_high.iloc[i], rtol=1e-8)

## 3. A/B/n: Šidák-adjusted significance decisions

Control + 3 challengers. `run_synthesis` compares each challenger with the Šidák-adjusted α;
R computes the same per-pair p-values independently and applies Šidák itself.

In [8]:
abn_visitors    = [12_000, 11_900, 12_100, 12_050]
abn_conversions = [   480,    540,    520,    455]
abn = FrequentistEngine().run_synthesis(ExperimentInput(
    visitors=abn_visitors, conversions=abn_conversions, confidence_level=conf,
))
abn_py = pd.DataFrame({"p_value": [r.p_value for r in abn],
                       "is_significant": [r.is_significant for r in abn]})
abn_alpha_py = apply_sidak(1 - conf, len(abn_visitors))
abn_py

,p_value,is_significant
0,0.039749,False
1,0.246790,False
2,0.368776,False


In [9]:
%%R -i abn_visitors -i abn_conversions -i conf -o abn_r -o abn_alpha_r
n <- unlist(abn_visitors); x <- unlist(abn_conversions)
m <- length(n) - 1
abn_alpha_r <- 1 - (1 - (1 - conf))^(1 / m)
abn_r <- do.call(rbind, lapply(2:length(n), function(i) {
  d <- data.frame(group = factor(c("control", "variant")), x = x[c(1, i)], n = n[c(1, i)])
  fit <- glm(cbind(x, n - x) ~ group, family = binomial(link = "identity"), data = d)
  p <- coef(summary(fit))["groupvariant", "Pr(>|z|)"]
  data.frame(p_value = p, is_significant = p < abn_alpha_r)
}))
abn_r

   p_value

 is_significant


1

 0.0397488

          FALSE


2

 0.2467897

          FALSE


3

 0.3687758

          FALSE

In [10]:
checks.add("Sidak alpha (4 arms)", abn_alpha_py, abn_alpha_r[0], rtol=1e-12)
for i in range(len(abn_py)):
    checks.add(f"A/B/n p-value challenger {i+1}", abn_py.p_value[i], abn_r.p_value.iloc[i], rtol=1e-6)
    checks.add_equal(f"A/B/n decision challenger {i+1}", bool(abn_py.is_significant[i]), bool(abn_r.is_significant.iloc[i]))

## 4. Monetary impact with different AOVs (delta method)

In [11]:
p_ctrl, n_ctrl, p_chal, n_chal = 320 / 10_000, 10_000, 365 / 10_050, 10_050
se_ctrl = np.sqrt(p_ctrl * (1 - p_ctrl) / n_ctrl)
se_chal = np.sqrt(p_chal * (1 - p_chal) / n_chal)
aov_ctrl, aov_chal, se_aov_ctrl, se_aov_chal = 82.0, 79.5, 2.1, 1.9
daily_visitors, period = 4_000, 183

money = {}
for alt in ["two-sided", "greater", "less"]:
    money[alt] = FrequentistEngine.estimate_monetary_impact_per_variant(
        p_ctrl=p_ctrl, se_ctrl=se_ctrl, aov_ctrl=aov_ctrl,
        p_chal=p_chal, se_chal=se_chal, aov_chal=aov_chal,
        daily_visitors=daily_visitors, alpha=1 - conf, alternative=Alt(alt),
        projection_period=period, se_aov_ctrl=se_aov_ctrl, se_aov_chal=se_aov_chal,
    )
pd.DataFrame(money)

,two-sided,greater,less
point_estimate,192745.432836,1.927454e+05,1.927454e+05
ci_low,-134780.452693,-8.212296e+04,-inf
ci_high,520271.318365,inf,4.676138e+05
daily_visitors,4000.000000,4.000000e+03,4.000000e+03
aov_ctrl,82.000000,8.200000e+01,8.200000e+01
aov_chal,79.500000,7.950000e+01,7.950000e+01
projection_period,183.000000,1.830000e+02,1.830000e+02


In [12]:
%%R -i p_ctrl -i p_chal -i se_ctrl -i se_chal -i aov_ctrl -i aov_chal -i se_aov_ctrl -i se_aov_chal -i daily_visitors -i period -i conf -o money_r
suppressPackageStartupMessages(library(msm))
mu  <- c(p_chal, aov_chal, p_ctrl, aov_ctrl)
cov <- diag(c(se_chal, se_aov_chal, se_ctrl, se_aov_ctrl)^2)
se_x <- deltamethod(~ x1 * x2 - x3 * x4, mu, cov)
x <- p_chal * aov_chal - p_ctrl * aov_ctrl
k <- daily_visitors * period
a <- 1 - conf
money_r <- data.frame(
  alternative = c("two-sided", "greater", "less"),
  point_estimate = x * k,
  ci_low  = c(x - qnorm(1 - a / 2) * se_x, x - qnorm(1 - a) * se_x, -Inf) * k,
  ci_high = c(x + qnorm(1 - a / 2) * se_x, Inf, x + qnorm(1 - a) * se_x) * k
)
money_r

 alternative

 point_estimate

     ci_low

  ci_high


1

   two-sided

... (9 outputs left out)

        less

       192745.4

       -Inf

 467613.8

In [13]:
for _, r in money_r.iterrows():
    m = money[r.alternative]
    for key in ["point_estimate", "ci_low", "ci_high"]:
        checks.add(f"monetary {key} | {r.alternative}", m[key], r[key], rtol=1e-8)

## 5. Lin's covariate adjustment (`run_lin_adjustment`)

User-level data, `y ~ variant * (pre − mean(pre))` with HC3 standard errors. R fits the same model
independently with `lm` and `sandwich::vcovHC(type = "HC3")`, then applies the z test, Šidák α and
interval itself. Scenarios: binary and continuous KPIs, 2 and 3 arms, heteroskedastic noise and a
heterogeneous effect, missing values, and a constant covariate (rank-deficient design, where R
reports the aliased coefficients as `NA`).

In [14]:
def lin_data(kind, n, labels, seed, const_pre=False, missing=False):
    rng = np.random.default_rng(seed)
    variant = rng.choice(labels, n)
    pre = np.zeros(n) if const_pre else rng.gamma(2.0, 3.0, n)
    arm = {lab: i for i, lab in enumerate(labels)}
    k = np.array([arm[v] for v in variant])
    if kind == "binary":
        prob = np.clip(0.08 + 0.01 * k + 0.004 * pre + 0.002 * k * pre, 0.01, 0.95)
        y = (rng.random(n) < prob).astype(float)
    else:
        y = rng.normal(40 + 2.0 * pre + 1.5 * k + 0.4 * k * pre, 5 + pre)
    df = pd.DataFrame({"variant": variant, "pre": pre, "y": y})
    if missing:
        df.loc[rng.choice(n, n // 50, replace=False), "y"] = np.nan
        df.loc[rng.choice(n, n // 80, replace=False), "pre"] = np.nan
    return df

lin_scenarios = {
    "binary 2 arms":        lin_data("binary", 8_000, ["control", "B"], 11),
    "binary 3 arms":        lin_data("binary", 9_000, ["control", "B", "C"], 12),
    "continuous 2 arms":    lin_data("continuous", 3_000, ["control", "B"], 13),
    "continuous 3 arms":    lin_data("continuous", 3_000, ["control", "B", "C"], 14),
    "missing values":       lin_data("continuous", 3_000, ["control", "B", "C"], 15, missing=True),
    "constant covariate":   lin_data("continuous", 2_000, ["control", "B"], 16, const_pre=True),
}

lin_rows = []
for scen, df in lin_scenarios.items():
    for alt in ["two-sided", "greater", "less"]:
        for r in FrequentistEngine.run_lin_adjustment(
            df, "y", "pre", alpha=1 - conf, control_label="control", alternative=Alt(alt)
        ):
            lin_rows.append(dict(scenario=scen, alternative=alt, variant=r["variant"],
                                 ate=r["absolute_lift"], se=r["std_err"], p_value=r["p_value"],
                                 ci_low=r["ci"][0], ci_high=r["ci"][1],
                                 relative_lift=r["relative_lift"], alpha=r["alpha"],
                                 is_significant=r["is_significant"]))
lin_py = pd.DataFrame(lin_rows)
lin_data_long = pd.concat([d.assign(scenario=s) for s, d in lin_scenarios.items()], ignore_index=True)
lin_py.head()

,scenario,alternative,variant,ate,se,p_value,ci_low,ci_high,relative_lift,alpha,is_significant
0,binary 2 arms,two-sided,B,0.035428,0.007153,7.308082e-07,0.021409,0.049447,0.356427,0.050000,True
1,binary 2 arms,greater,B,0.035428,0.007153,3.654041e-07,0.023663,inf,0.356427,0.050000,True
2,binary 2 arms,less,B,0.035428,0.007153,9.999996e-01,-inf,0.047193,0.356427,0.050000,False
3,binary 3 arms,two-sided,B,0.018749,0.008180,2.190153e-02,0.000455,0.037043,0.183439,0.025321,True
4,binary 3 arms,two-sided,C,0.043670,0.008392,1.951933e-07,0.024902,0.062439,0.427267,0.025321,True


In [15]:
%%R -i lin_data_long -i conf -o lin_r
suppressPackageStartupMessages(library(sandwich))
a <- 1 - conf
lin_r <- do.call(rbind, lapply(unique(lin_data_long$scenario), function(s) {
  d <- lin_data_long[lin_data_long$scenario == s, ]
  d$cc <- d$pre - mean(d$pre, na.rm = TRUE)
  d$variant <- relevel(factor(d$variant), ref = "control")
  fit <- lm(y ~ variant * cc, data = d)
  V <- vcovHC(fit, type = "HC3")
  m <- nlevels(d$variant) - 1
  a_adj <- 1 - (1 - a)^(1 / m)
  do.call(rbind, lapply(levels(d$variant)[-1], function(lab) {
    nm <- paste0("variant", lab)
    est <- coef(fit)[[nm]]; se <- sqrt(V[nm, nm]); z <- est / se
    do.call(rbind, lapply(c("two-sided", "greater", "less"), function(alt) {
      p <- switch(alt, "two-sided" = 2 * pnorm(-abs(z)),
                       "greater" = pnorm(z, lower.tail = FALSE), "less" = pnorm(z))
      ci <- switch(alt,
        "two-sided" = est + c(-1, 1) * qnorm(1 - a_adj / 2) * se,
        "greater"   = c(est - qnorm(1 - a_adj) * se, Inf),
        "less"      = c(-Inf, est + qnorm(1 - a_adj) * se))
      data.frame(scenario = s, alternative = alt, variant = lab, ate = est, se = se,
                 p_value = p, ci_low = ci[1], ci_high = ci[2],
                 relative_lift = est / coef(fit)[["(Intercept)"]],
                 alpha = a_adj, is_significant = p < a_adj)
    }))
  }))
}))

In [16]:
lin_cmp = lin_py.merge(lin_r, on=["scenario", "alternative", "variant"], suffixes=("_py", "_r"))
assert len(lin_cmp) == len(lin_py) == len(lin_r)
for _, row in lin_cmp.iterrows():
    tag = f"{row.scenario} | {row.variant} | {row.alternative}"
    checks.add(f"Lin ATE      {tag}", row.ate_py, row.ate_r, rtol=1e-8)
    checks.add(f"Lin HC3 SE   {tag}", row.se_py, row.se_r, rtol=1e-8)
    checks.add(f"Lin p-value  {tag}", row.p_value_py, row.p_value_r, atol=1e-12, rtol=1e-8)
    checks.add(f"Lin CI low   {tag}", row.ci_low_py, row.ci_low_r, rtol=1e-8)
    checks.add(f"Lin CI high  {tag}", row.ci_high_py, row.ci_high_r, rtol=1e-8)
    checks.add(f"Lin rel lift {tag}", row.relative_lift_py, row.relative_lift_r, rtol=1e-8)
    checks.add(f"Lin Sidak a  {tag}", row.alpha_py, row.alpha_r, rtol=1e-12)
    checks.add_equal(f"Lin decision {tag}", bool(row.is_significant_py), bool(row.is_significant_r))

## Summary

In [17]:
checks.report()

384 checks, 384 passed, 0 failed

Largest differences:
                              check  hexkit       R    abs_diff              tolerance
            monetary ci_high | less  467614  467614 1.74623e-10 atol=1e-08, rtol=1e-08
        monetary ci_low | two-sided -134780 -134780 1.16415e-10 atol=1e-08, rtol=1e-08
       monetary ci_high | two-sided  520271  520271 1.16415e-10 atol=1e-08, rtol=1e-08
          monetary ci_low | greater  -82123  -82123 8.73115e-11 atol=1e-08, rtol=1e-08
monetary point_estimate | two-sided  192745  192745 2.91038e-11 atol=1e-08, rtol=1e-08


,check,hexkit,R,abs_diff,tolerance,pass
0,SE typical | two-sided | phi=1.0,0.002565,0.002565,4.336809e-19,"atol=1e-08, rtol=1e-06",True
1,p-value typical | two-sided | phi=1.0,0.092283,0.092283,9.714451e-17,"atol=1e-09, rtol=1e-06",True
2,CI low typical | two-sided | phi=1.0,-0.000709,-0.000709,1.734723e-18,"atol=1e-08, rtol=1e-06",True
3,CI high typical | two-sided | phi=1.0,0.009346,0.009346,3.469447e-18,"atol=1e-08, rtol=1e-06",True
4,SE typical | two-sided | phi=0.6,0.001987,0.001987,0.000000e+00,"atol=1e-08, rtol=1e-06",True
...,...,...,...,...,...,...
379,Lin CI low constant covariate | B | less,-inf,-inf,0.000000e+00,"atol=1e-08, rtol=1e-08",True
380,Lin CI high constant covariate | B | less,1.843336,1.843336,2.686740e-14,"atol=1e-08, rtol=1e-08",True
381,Lin rel lift constant covariate | B | less,0.036634,0.036634,4.857226e-16,"atol=1e-08, rtol=1e-08",True
382,Lin Sidak a constant covariate | B | less,0.05,0.05,0.000000e+00,"atol=1e-08, rtol=1e-12",True
